# Inpainting tables for paper
- Metrics: PSNR_masked, SSIM_masked
- Shots: Few shot=5, Many shot=15
- Averaged across cvfold 0/1/2 within each datatype,trlim
- Exports csv and latex with best per column bolded

## Setup

In [1]:
# imports
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import re
from typing import Dict, List, Optional, Tuple


In [2]:
# config

OUTDIR = Path("/midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables")
OUTDIR.mkdir(parents=True, exist_ok=True)

FOLDS = [0, 1, 2]
TRLIMS = [5, 15]          # few-shot, many-shot
DECIMALS = 2
TABCOLSEP_PT = 1          # narrow columns (MICCAI-safe)
ARRAYSTRETCH = 1.05

DTYPES_GT = ["amyloid_plaque_patches", "cell_nucleus_patches", "vessels_patches"]
DTYPE_CANON = {
    "amyloid_plaque_patches": "Amyloid Plaque",
    "cell_nucleus_patches": "Cell Nucleus",
    "vessels_patches": "Vessels",
}

INPAINT_ROOTS: Dict[str, Path] = {
    "UNet Img+Txt (t)": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_super_sweep2"),
    "UNet Img+Txt (nt)": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_super_sweep2_ntc"),
    "UNet Img": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_bright_sweep_26_ntc"),
    "UNet Rand": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_unet_random2_ntc"),
    "SwinUNETR Img+Txt (t)": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_autumn_sweep_27_subset"),
    "SwinUNETR Img+Txt (nt)": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_autumn_sweep_27_ntc_subset"),
    "SwinUNETR Img": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_expert_sweep_31_ntc_subset"),
    "SwinUNETR Rand": Path("/midtier/paetzollab/scratch/ads4015/temp_selma_inpaint_preds_random_ntc_subset"),
}

MODEL_ORDER = [
    "UNet Img+Txt (t)",
    "SwinUNETR Img+Txt (t)",
    "UNet Img+Txt (nt)",
    "SwinUNETR Img+Txt (nt)",
    "UNet Img",
    "SwinUNETR Img",
    "UNet Rand",
    "SwinUNETR Rand",
]

SHOT_LABEL = {5: "Few-shot", 15: "Many-shot"}  # rename "Many-shot" -> "Multi-shot" if you prefer

# wrap long row names to conserve width (requires \usepackage{makecell})
ROWNAME_WRAP = {
    "SwinUNETR Img+Txt (t)":  r"\makecell[l]{SwinUNETR\\Img+Txt (t)}",
    "SwinUNETR Img+Txt (nt)": r"\makecell[l]{SwinUNETR\\Img+Txt (nt)}",
    "SwinUNETR Img":         r"\makecell[l]{SwinUNETR\\Img}",
    "SwinUNETR Rand":        r"\makecell[l]{SwinUNETR\\Rand}",
    "UNet Img+Txt (t)":      r"\makecell[l]{UNet\\Img+Txt (t)}",
    "UNet Img+Txt (nt)":     r"\makecell[l]{UNet\\Img+Txt (nt)}",
}


## Helper functions

In [3]:
# safe mean
def safe_mean(xs: List[float]) -> float:
    xs = [x for x in xs if x is not None and np.isfinite(x)]
    return float(np.mean(xs)) if xs else float("nan")


# pick run dir for fold and trlim
def pick_run_dir_for_fold_trlim(dtype_dir: Path, fold: int, trlim: int) -> Optional[Path]:

    if not dtype_dir.exists():
        return None
    pat = re.compile(rf"cvfold{fold}_trl(?:im|m){trlim}(?:_|$)")
    run_dirs = [d for d in dtype_dir.iterdir() if d.is_dir() and pat.search(d.name)]
    if not run_dirs:
        return None
    run_dirs.sort(key=lambda d: d.stat().st_mtime, reverse=True)
    return run_dirs[0]


# read metrics_test.csv from run dir
def read_metrics_test_csv(run_dir: Path) -> Optional[pd.DataFrame]:

    if run_dir is None or not run_dir.exists():
        return None

    c1 = run_dir / "preds" / "metrics_test.csv"
    if c1.exists():
        return pd.read_csv(c1)

    hits = list(run_dir.rglob("metrics_test.csv"))
    if not hits:
        return None
    hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return pd.read_csv(hits[0])


# evaluate one fold + trlim (returns mean PSNR_masked and mean SSIM_masked over 2 test patches from each fold)
def eval_fold_trlim(root: Path, dtype_gt: str, fold: int, trlim: int) -> Tuple[float, float]:

    dtype_dir = root / "preds" / dtype_gt
    run_dir = pick_run_dir_for_fold_trlim(dtype_dir, fold=fold, trlim=trlim)
    dfm = read_metrics_test_csv(run_dir)
    if dfm is None or dfm.empty:
        return (float("nan"), float("nan"))

    psnr = pd.to_numeric(dfm.get("psnr_masked", pd.Series(dtype=float)), errors="coerce")
    ssim = pd.to_numeric(dfm.get("ssim_masked", pd.Series(dtype=float)), errors="coerce")
    return (float(psnr.mean(skipna=True)), float(ssim.mean(skipna=True)))


# evaluate one datatype + trlim (averaged across folds 0/1/2)
def eval_model_dtype_trlim(root: Path, dtype_gt: str, trlim: int) -> Dict[str, float]:
    """
    Average across cvfold0/1/2 for a given datatype + trlim.
    """
    psnrs, ssims = [], []
    for fold in FOLDS:
        p, s = eval_fold_trlim(root, dtype_gt, fold=fold, trlim=trlim)
        psnrs.append(p)
        ssims.append(s)
    return {"PSNR": safe_mean(psnrs), "SSIM": safe_mean(ssims)}


# bold best per numeric column (LaTeX)
def latex_bold_max_per_column(df_num: pd.DataFrame, decimals: int = 2, na_rep: str = "—") -> pd.DataFrame:
    fmt = f"{{:.{decimals}f}}"
    out = pd.DataFrame(index=df_num.index, columns=df_num.columns, dtype=object)
    for col in df_num.columns:
        s = pd.to_numeric(df_num[col], errors="coerce")
        maxv = s.max(skipna=True)
        col_out = []
        for v in s.values:
            if not np.isfinite(v):
                col_out.append(na_rep)
            else:
                txt = fmt.format(float(v))
                if np.isfinite(maxv) and np.isclose(v, maxv, rtol=0, atol=1e-12):
                    txt = rf"\textbf{{{txt}}}"
                col_out.append(txt)
        out[col] = col_out
    return out


# replace first tabular colspec in LaTeX string
def replace_first_tabular_colspec(latex_str: str, colspec: str) -> str:
    return re.sub(
        r"\\begin\{tabular\}\{[^}]*\}",
        rf"\\begin{{tabular}}{{{colspec}}}",
        latex_str,
        count=1,
    )


# inject compact table formatting into LaTeX string
def inject_table_formatting(
    latex_str: str,
    add_centering: bool = True,
    fontsize_cmd: str = r"\fontsize{8}{9}\selectfont",
    tabcolsep_pt: float = TABCOLSEP_PT,
    arraystretch: float = ARRAYSTRETCH,
) -> str:
    lines = latex_str.splitlines()
    out = []
    injected = False
    for line in lines:
        out.append(line)
        s = line.strip()
        if (not injected) and (s.startswith(r"\begin{table}") or s.startswith(r"\begin{sidewaystable}")):
            if add_centering:
                out.append(r"\centering")
            out.append(fontsize_cmd)
            out.append(rf"\setlength{{\tabcolsep}}{{{tabcolsep_pt}pt}}")
            out.append(rf"\renewcommand{{\arraystretch}}{{{arraystretch}}}")
            injected = True
    return "\n".join(out)


In [4]:
# build table

# build records
records = []
for model in MODEL_ORDER:
    root = INPAINT_ROOTS[model]
    row = {"Model": model}
    for dtype_gt in DTYPES_GT:
        dtype_name = DTYPE_CANON[dtype_gt]
        for trlim in TRLIMS:
            shot = SHOT_LABEL[trlim]
            m = eval_model_dtype_trlim(root, dtype_gt, trlim=trlim)
            row[(dtype_name, shot, "PSNR")] = m["PSNR"]
            row[(dtype_name, shot, "SSIM")] = m["SSIM"]
    records.append(row)

df_inpaint = pd.DataFrame.from_records(records).set_index("Model")
df_inpaint.columns = pd.MultiIndex.from_tuples(df_inpaint.columns)

# save numeric CSV
csv_out = OUTDIR / "inpaint_results.csv"
df_inpaint.round(6).to_csv(csv_out)
print(f"[Saved] {csv_out}")

# latex export (bold best per column)
df_num = df_inpaint.round(DECIMALS)
df_tex = latex_bold_max_per_column(df_num, decimals=DECIMALS, na_rep="—")

# Wrap long row names (requires \usepackage{makecell} in paper)
df_tex = df_tex.copy()
df_tex.index = [ROWNAME_WRAP.get(str(i), str(i)) for i in df_tex.index]

latex_str = df_tex.to_latex(
    escape=False,
    multicolumn=True,
    multirow=True,
    caption=(
        "Inpainting performance (masked PSNR and masked SSIM) for few-shot (samples=5) and many-shot (samples=15). "
        "Values are averaged across 3 CV folds."
    ),
    label="tab:inpaint_results",
    bold_rows=False,
    longtable=False,
    index=True,
)

# center datatype headers (each datatype spans 4 cols: Few(PSNR,SSIM) + Many(PSNR,SSIM))
for pretty in DTYPE_CANON.values():
    latex_str = latex_str.replace(rf"\multicolumn{{4}}{{r}}{{{pretty}}}", rf"\multicolumn{{4}}{{c}}{{{pretty}}}")

# center shot headers across PSNR/SSIM
latex_str = latex_str.replace(r"\multicolumn{2}{r}{Few-shot}",  r"\multicolumn{2}{c}{Few-shot}")
latex_str = latex_str.replace(r"\multicolumn{2}{r}{Many-shot}", r"\multicolumn{2}{c}{Many-shot}")

# inject compact formatting (MICCAI-safe)
latex_str = inject_table_formatting(latex_str)

# save LaTeX
tex_out = OUTDIR / "inpaint_results.tex"
tex_out.write_text(latex_str)
print(f"[Saved] {tex_out}")

# display
display(df_inpaint.round(DECIMALS))


[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/inpaint_results.csv
[Saved] /midtier/paetzollab/scratch/ads4015/lsm_fm_paper/tables/inpaint_results.tex


Amyloid Plaque                       Cell Nucleus  \
                             Few-shot       Many-shot           Few-shot   
                                 PSNR  SSIM      PSNR  SSIM         PSNR   
Model                                                                      
UNet Img+Txt (t)                31.17  0.98     33.67  0.99        30.22   
SwinUNETR Img+Txt (t)           34.50  0.99     38.02  0.99        32.99   
UNet Img+Txt (nt)               33.48  0.99     35.41  0.99        23.77   
SwinUNETR Img+Txt (nt)          34.80  0.99     38.91  0.99        33.41   
UNet Img                        35.51  0.99     39.03  0.99        23.71   
SwinUNETR Img                   32.42  0.98     34.82  0.99        32.78   
UNet Rand                       33.36  0.99     36.12  0.99        27.54   
SwinUNETR Rand                  36.95  0.99     38.45  0.99        32.53   

                                              Vessels                        
                             Many-shot       Few-shot       Many-shot        
                        SSIM      PSNR  SSIM     PSNR  SSIM      PSNR  SSIM  
Model                                                                        
UNet Img+Txt (t)        0.94     31.66  0.95    31.08  0.97     32.18  0.97  
SwinUNETR Img+Txt (t)   0.96     34.04  0.97    37.41  0.99     38.14  0.99  
UNet Img+Txt (nt)       0.92     27.34  0.94    34.14  0.98     37.69  0.99  
SwinUNETR Img+Txt (nt)  0.97     34.52  0.97    37.63  0.99     38.78  0.99  
UNet Img                0.92     23.71  0.92    34.54  0.99     37.94  0.99  
SwinUNETR Img           0.96     34.09  0.97    37.81  0.99     38.22  0.99  
UNet Rand               0.93     31.49  0.95    33.39  0.98     36.95  0.99  
SwinUNETR Rand          0.96     33.81  0.97    37.85  0.99     39.68  1.00